In [ ]:
%pip install fiona shapely pyproj

In [ ]:
import fiona
from shapely.geometry import box, mapping
from fiona.crs import CRS

# ------------------------------------------------------------------
# Output shapefile path (folder must already exist)
# ------------------------------------------------------------------
out_shp = "/g/data/jk72/ae8456/epsg3031_regions.shp"

# ------------------------------------------------------------------
# Define regions: (name, min_e, min_n, max_e, max_n)
# Units: meters, EPSG:3031
# ------------------------------------------------------------------
regions = [
    ("DML", -646140, 1640000, 932275, 2354251),
    
    ("DML_to_Cape_Ann", 932275, 1520745, 2261917, 2046358),
    
    ("Mawson Coast", 2059059, 733836, 2477921, 1520453),
    ("Pub to Shackelton", 2085674, -228550, 2841953, 724017),
    
    ("Shackelton to MoscowU", 2119561, -1363887, 2871187, -199935),
    
    ("MoscowU to Dibble", 1842537, -1891859, 2350990, -1334150),
    
    ("Dibble to Cape Adare", 281740, -2345499, 2031736, -1843046),
    
    ("Victoria Land", -15069, -1963850, 539119, -1251802),

    ("Sulzberger Ice Shelf to Dean Island", -1349852, -1405803, -255911, -1081185),

    ("Dean Island  to Pine Island Bay", -1835944,-1191356, -1347852,-296400),

    ("Cape Flying Fish to Scorseby Head", -2025741, -457109, -1738355, 375962),
    
    ("Scorseby Head (Across Peninsula) to Larsen C", -2556743, 344111, -1391030, 1325911),
    
    ("Northern Peninsula", -2690279, 1253973, -2290420, 1734501),

    ("Southern Weddell Sea to Brunt Ice Shelf", -1277006, 800205, -630969, 1744333),
    
]

# ------------------------------------------------------------------
# Shapefile schema
# ------------------------------------------------------------------
schema = {
    "geometry": "Polygon",
    "properties": {
        "region": "str"
    }
}

# ------------------------------------------------------------------
# CRS: Antarctic Polar Stereographic
# ------------------------------------------------------------------
crs = CRS.from_epsg(3031)

# ------------------------------------------------------------------
# Write shapefile
# ------------------------------------------------------------------
with fiona.open(
    out_shp,
    mode="w",
    driver="ESRI Shapefile",
    schema=schema,
    crs=crs,
) as shp:

    for name, min_e, min_n, max_e, max_n in regions:
        polygon = box(min_e, min_n, max_e, max_n)

        shp.write({
            "geometry": mapping(polygon),
            "properties": {
                "region": name
            }
        })

print(f"Shapefile written: {out_shp}")

In [ ]:
from pyproj import Transformer
transformer = Transformer.from_crs(
    "EPSG:3031",
    "EPSG:4326",
    always_xy=True
)

# ------------------------------------------------------------------
# Compute lon/lat bounds
# ------------------------------------------------------------------
region_bounds_ll = {}

for name, min_e, min_n, max_e, max_n in regions:

    corners = [
        (min_e, min_n),
        (min_e, max_n),
        (max_e, min_n),
        (max_e, max_n),
    ]

    lons, lats = [], []

    for x, y in corners:
        lon, lat = transformer.transform(x, y)
        lons.append(lon)
        lats.append(lat)

    region_bounds_ll[name] = {
        "min_lon": min(lons),
        "max_lon": max(lons),
        "min_lat": min(lats),
        "max_lat": max(lats),
    }

# ------------------------------------------------------------------
# Print results
# ------------------------------------------------------------------
for name, b in region_bounds_ll.items():
    print(f"{name}")
    print(f"  Lon: {b['min_lon']:.2f} to {b['max_lon']:.2f}")
    print(f"  Lat: {b['min_lat']:.2f} to {b['max_lat']:.2f}")
    print()

In [ ]:
min_lat = min(region['min_lat'] for region in region_bounds_ll.values())
max_lat = max(region['max_lat'] for region in region_bounds_ll.values())

print("Overall min_lat:", min_lat)
print("Overall max_lat:", max_lat)